# Quickstart: cross-group composition (verbose × formal)

Demonstrates Phase 4's `compose([probe_a, probe_b])` — steering with vectors from *different* `ConceptGroup`s simultaneously. Because each group has its own neutral reference, directions from different groups live in independent coordinate frames and can be linearly combined at inference time. This is a capability `repeng` and CAA do not provide.

We build two single-concept groups (verbose and formal) from the bundled seed data, sweep both, and compose their best probes for joint steering.

In [ ]:
import os

os.environ.setdefault("TRANSFORMERLENS_ALLOW_MPS", "1")

from pathlib import Path

from steerkit import (
    Concept,
    ConceptGroup,
    compose,
    load,
    load_pairs_jsonl,
    sweep,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

verb_pairs = load_pairs_jsonl(REPO_ROOT / "examples" / "data" / "verbosity.jsonl")
form_pairs = load_pairs_jsonl(REPO_ROOT / "examples" / "data" / "formality.jsonl")
print(f"verbosity: {len(verb_pairs)} pairs   formality: {len(form_pairs)} pairs")

In [ ]:
verb_group = ConceptGroup(
    name="verbosity",
    relationship="axes",
    neutral_reference="Respond as concisely as possible",
    concepts=[
        Concept(
            name="verbose",
            description="long, expansive, with many examples",
            contrast_pairs=verb_pairs,
        )
    ],
)
form_group = ConceptGroup(
    name="formality",
    relationship="axes",
    neutral_reference="Respond casually, like talking to a friend",
    concepts=[
        Concept(
            name="formal",
            description="formal, polished, professional, no contractions",
            contrast_pairs=form_pairs,
        )
    ],
)

In [ ]:
model = load("EleutherAI/pythia-160m")
verb_fit = sweep(verb_group, model, cache_dir=REPO_ROOT / "cache")
form_fit = sweep(form_group, model, cache_dir=REPO_ROOT / "cache")
print(f"verbose best: layer {verb_fit['verbose'].layer} (depth {verb_fit['verbose'].normalized_depth:.2f})")
print(f"formal best:  layer {form_fit['formal'].layer} (depth {form_fit['formal'].normalized_depth:.2f})")

## Compose the two probes

`compose` returns a `CompositeProbe` whose `.steer(...)` installs hooks at every constituent probe's layer. Probes targeting the same hook are folded into a single combined direction; probes at different layers each get their own hook.

Adjust `weights` to control the relative push from each concept.

In [ ]:
test_prompt = "Tell me about your morning routine."
verb_only = verb_fit['verbose'].steer(model, test_prompt, max_new_tokens=40)
form_only = form_fit['formal'].steer(model, test_prompt, max_new_tokens=40)

verb_form = compose([verb_fit['verbose'], form_fit['formal']])
both = verb_form.steer(model, test_prompt, max_new_tokens=40)

verb_form_weighted = compose([verb_fit['verbose'], form_fit['formal']], weights=[1.5, 0.5])
more_verb = verb_form_weighted.steer(model, test_prompt, max_new_tokens=40)

print("verbose only            :", verb_only)
print("formal only             :", form_only)
print("verbose + formal (1:1)  :", both)
print("verbose + formal (3:1)  :", more_verb)

## Visualize where each probe peaks

If the two probes choose layers far apart in the network, the composition installs hooks at *both* layers. If they choose the same layer, the hook combines them into one cumulative direction.

In [ ]:
import matplotlib.pyplot as plt

fig = verb_fit.plot_layer_selection("verbose", by_classifier="cohens_d_logistic", x_axis="normalized_depth", title="verbose")
plt.show()
fig = form_fit.plot_layer_selection("formal", by_classifier="cohens_d_logistic", x_axis="normalized_depth", title="formal")
plt.show()